# 🎙️ SLAM-ASR Turkish: Common Voice 27.0 Benchmark Notebook

Bu notebook, **SLAM-LLM** (Speech-Language Model Framework) metodolojisini birebir uygulayarak **Mozilla Common Voice 27.0 Türkçe** veri setinde state-of-the-art (SOTA) ASR başarımını elde etmeniz için tasarlanmıştır.

### ⚡ Özellikler:
- Kodlar doğrudan **GitHub** üzerinden Colab ortamına indirilir.
- Veri seti `/content/data` klasöründe saklanır; proje yeniden klonlansa veya güncellense dahi **hazırlanan veriler silinmez**.
- Veri seti doğrudan **Mozilla Data Collective (MDC)** API üzerinden `CV_API` anahtarı ile otomatik indirilir.
- Eğitilen model ağırlıkları ve checkpoint'ler doğrudan **Hugging Face Hub** hesabınıza otomatik yüklenir.
- **Speech Encoder:** `openai/whisper-large-v3` (Frozen)
- **Projector:** 5x Frame Concatenation Linear Projector (Trainable)
- **LLM Backbone:** `Qwen/Qwen2.5-7B-Instruct` (LoRA Fine-tuning)

---

## 1. Donanım & Ekran Kartı Kontrolü (GPU Check)

In [ ]:
!nvidia-smi

## 2. Projenin GitHub'dan Colab Ortamına İndirilmesi

In [ ]:
%cd /content
!rm -rf slam-asr
!git clone https://github.com/gokhanersoy/slam-asr.git
%cd /content/slam-asr/commonvoice_tr

## 3. Gerekli Kütüphanelerin Kurulumu

In [ ]:
!pip install -q -r requirements.txt

## 4. API Anahtarları ve Yetkilendirme 🔑

- Colab Secrets (Gizli Anahtarlar) alanına eklediğiniz `CV_API` (Mozilla Data Collective) ve `HF_TOKEN` (Hugging Face) otomatik yüklenir.

In [ ]:
import os
try:
    from google.colab import userdata
    cv_api = userdata.get('CV_API')
    if cv_api:
        os.environ['CV_API'] = cv_api
        print("✅ CV_API Mozilla Data Collective anahtarı yüklendi!")

    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ HF_TOKEN Hugging Face anahtarı yüklendi!")
except Exception as e:
    print(f"Bilgi: {e}")

## 5. Türkçe Common Voice 27.0 Veri Setinin İndirilmesi & Hazırlanması
Mozilla Data Collective API üzerinden `CV_API` ile Common Voice 27.0 Türkçe arşivi indirilir, `/content/data` klasörüne ayıklanır, 16kHz WAV formatına dönüştürülür ve JSONL manifests dosyaları (`train.jsonl`, `val.jsonl`, `test.jsonl`) üretilir.

In [ ]:
# Hızlı deneme yapmak isterseniz --max_train_samples 10000 ekleyebilirsiniz.
# Tüm veri seti ile eğitmek için parametresiz çalıştırın:
!python prepare_data.py --output_dir /content/data
#!python prepare_data.py --output_dir /content/data --max_train_samples 1000 --max_val_samples 200 --max_test_samples 200

## 6. Model Eğitimi (SLAM-ASR Fine-Tuning)
Whisper Large V3 dondurulur, 2 katmanlı Linear Projector katmanı eğitilir ve Qwen2.5-7B-Instruct modeline LoRA uygulanır.

In [ ]:
!python train.py train_config.num_epochs=5 train_config.batch_size_training=2 train_config.gradient_accumulation_steps=8
#!python train.py train_config.num_epochs=3 train_config.validation_interval=50

## 7. Test Setinde Değerlendirme (WER & CER Hesaplama)

In [ ]:
!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --output_file evaluation_results.json
#!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --max_samples 100

## 8. Eğitilen Modeli Hugging Face Hub'a Yükleme 🚀
Eğitilen model ağırlıklarını tek komutla kendi Hugging Face hesabınıza yükleyin.

In [ ]:
# Kendi Hugging Face Kullanıcı Adınızı Yazın (Örn: gokhanersoy/slam-asr-tr-commonvoice)
HF_REPO_ID = "gokhanersoy/slam-asr-tr-commonvoice"

!python push_to_hub.py --repo_id {HF_REPO_ID} --checkpoint_dir checkpoints/slam_qwen_tr

## 9. Örnek Deşifre Playground (Single Audio Inference)

In [ ]:
import torch
import whisper
from omegaconf import OmegaConf
from src.models.slam_model import model_factory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = OmegaConf.load("conf/prompt.yaml")

# Modeli yükle
model, tokenizer = model_factory(
    train_config=cfg.train_config,
    model_config=cfg.model_config,
    ckpt_path="checkpoints/slam_qwen_tr/best_checkpoint.pt"
)
model.to(device)
model.eval()

# Örnek Ses Dosyası Yolu
sample_audio_path = "/content/data/audio/test/test_000000.wav"

# Log Mel Spectrogram
audio_raw = whisper.load_audio(sample_audio_path)
audio_raw = whisper.pad_or_trim(audio_raw)
audio_mel = whisper.log_mel_spectrogram(audio_raw, n_mels=128).permute(1, 0).unsqueeze(0).to(device)

# Tahmin Al
prediction = model.generate_transcription(audio_mel=audio_mel)
print(f"🎙️ Tahmin Edilen Metin: {prediction}")
